# Calculations and demos of the particle and field handling

In [ ]:
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

### Auxiliary functions

In [ ]:
seed = 137

In [ ]:
def generate_normal(size, *, loc=0, scale=1, seed=137):
    rng = np.random.default_rng(seed=seed)
    return rng.normal(loc=loc, scale=scale, size=size)

def generate_integer(size, *, low=0, high=10, seed=137):
    rng = np.random.default_rng(seed=seed)
    return rng.integers(low=low, high=high, size=size)

def generate_uniform(size, *, low=0, high=1, seed=137):
    rng = np.random.default_rng(seed=seed)
    return rng.uniform(low=low, high=high, size=size)

### White noise generation

In [ ]:
Lbox = [1000, 400, 200]
x = generate_uniform(size=(1000, 3)) * np.array(Lbox)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, color='0.3', s=4**2, ec='none', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')

plt.show()

In [ ]:
def cubic_voxels(nmesh, Lbox, silent=False):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.
    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on the shortest dimension of the cuboid and
    scales the other dimensions accordingly.

    Parameters
    ----------
    nmesh : int
        Number of voxels in the shortest dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz] or a single
        float value for a cubic box.
    silent : bool
        If True, suppresses output messages.

    Returns
    -------
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    mesh : tuple of int
        A tuple containing the number of voxels in each dimension (Nx, Ny, Nz).
    '''
    if not isinstance(Lbox, (list, tuple)):
        Lbox = (Lbox,) * 3
    ref_L = np.min(Lbox)
    mesh = np.ceil(Lbox / (ref_L / nmesh)).astype(int)
    mesh = (mesh + mesh % 2).astype(int)  # Ensure even number of voxels
    if not silent:
        print('Mesh: Nx={}, Ny={}, Nz={}'.format(*mesh))
    dk = ref_L / mesh[Lbox.index(ref_L)]
    if not silent:
        print(f'Step size: {dk}')
    return dk, mesh

In [ ]:
nmesh = 8
Lbox = [100, 100, 50]
dk, nvox = cubic_voxels(nmesh, Lbox)

mesh = [np.linspace(dk/2, L-dk/2, m, endpoint=True) for L, m in zip(Lbox, nvox)]
grid = np.meshgrid(*mesh, indexing='ij')
field = generate_normal(size=nvox)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]
    
    ax.scatter(x1_s, x2_s, color='0.3', s=6**2, ec='none')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]
    
    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.scatter(x1_s, x2_s, c=field_s, s=10**2, ec='none')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.pcolormesh(x1_s, x2_s, field_s, shading='auto')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

### Field interpolation to particle positions

In [ ]:
from scipy.interpolate import RegularGridInterpolator

In [ ]:
def interpolate_field(x, field, Lbox, method='linear'):
    r'''
    Interpolate a grid-based field onto particle positions using periodic
    boundaries.

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Particle positions in the simulation box.
    field : ndarray
        The grid-based field (e.g. a displacement field) defined on a
        regular grid.
    Lbox : float
        The simulation box size.
    method : str, optional
        The interpolation method to use. This can be 'linear', 'nearest',
        or 'cubic'. The default is 'linear'.

    Returns
    -------
    interp_values : ndarray of shape (N,)
        Field values interpolated at the particle positions.
    '''
    if not isinstance(Lbox, (list, tuple)):
        Lbox = (Lbox,) * 3
    nvox = field.shape
    dk = np.array(Lbox) / np.array(nvox)  # Assuming cubical voxels
    mesh = [np.linspace(di/2, L-di/2, n, endpoint=True) for L, n, di in zip(Lbox, nvox, dk)]
    interpolator = RegularGridInterpolator(
        tuple(mesh),
        field,
        method=method,
        bounds_error=False,
        fill_value=None  # Extrapolate using periodic wrapping if needed
    )
    return interpolator(np.mod(x, Lbox))

In [ ]:
nmesh = 32
Lbox = [100, 100, 50]
dk, nvox = cubic_voxels(nmesh, Lbox)

field = generate_normal(size=nvox)
mesh = [np.linspace(dk/2, L-dk/2, m, endpoint=True) for L, m in zip(Lbox, nvox)]
grid = np.meshgrid(*mesh, indexing='ij')

x = generate_uniform(size=(2000, 3)) * np.array(Lbox)
field_interp = interpolate_field(x, field, Lbox)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]
    
    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.scatter(x1_s, x2_s, c=field_s, s=4**2, ec='none')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.pcolormesh(x1_s, x2_s, field_s, shading='auto')
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, c=field_interp, s=4**2, ec='none', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()